# Approximate Riemann solvers

Here we write 4 approximate solvers and compare them on the Sod porblem, with everything else in the code left untouched.

* **Part I.** Lax-Friedrichs, HLL, HLLC and Roe Solvers

* **Part II.** Sod problem

Each solver takes the two states at an interface and returns the flux there:

$$\mathbf{F}_{i+1/2} = \mathcal{R}\left(\mathbf{q}_{L},\, \mathbf{q}_{R}\right)$$

so they are interchangeable: the FV loop receives the solver as an argument.

**References**: E. F. Toro, *Riemann Solvers and Numerical Methods for Fluid Dynamics*, 2nd ed., Chapters 10 and 11.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

## Part I. Solvers:

### 1. Variables and fluxes:

We re-use some functions from nb18, with $\mathbf{U} = (\rho,\ \rho u,\ E)$ and $\mathbf{q} = (\rho,\ u,\ p)$.

In [ ]:
# Fix adiabatic index
gamma = 1.4

In [ ]:
def sound_speed(rho, p, gamma = 1.4):
    """
    Speed of sound for an ideal gas
    """
    return np.sqrt(gamma*p/rho)

def conserved(q, gamma = 1.4):
    """
    From q = (rho, u, p) to U = (rho, rho*u, E).
    """

    rho, u, p = q

    # E is the total energy per unit volume: internal + kinetic
    return np.array([rho, rho*u, p/(gamma - 1.) + 0.5*rho*u**2])

def primitive(U, gamma = 1.4):
    """
    From U = (rho, rho*u, E) to q = (rho, u, p).
    """

    rho = U[0]

    # Velocity from the momentum
    u = U[1]/rho

    # Pressure from the ideal-gas law
    p = (gamma - 1.)*(U[2] - 0.5*rho*u**2)

    return np.array([rho, u, p])

def flux(q, gamma = 1.4):
    """
    Euler flux in primitive variables.
    """

    rho, u, p = q

    # Total energy per unit volume
    E = p/(gamma - 1.) + 0.5*rho*u**2

    # F = (rho*u, rho*u^2 + p, u*(E + p))
    return np.array([rho*u, rho*u**2 + p, u*(E + p)])

### 2. Signal speeds:

Every solver below starts from bounds for the slowest and the fastest wave leaving the interface.


We use the Davis estimate:

$$S_L = \min\left(u_L - c_L,\; u_R - c_R\right), \qquad\qquad S_R = \max\left(u_L + c_L,\; u_R + c_R\right)$$

In [ ]:
def signal_speeds(qL, qR, gamma = 1.4):
    """
    Lower and upper bounds for the wave speeds in the fan.
    """
    # Sound speeds
    cL = sound_speed(qL[0], qL[2], gamma)
    cR = sound_speed(qR[0], qR[2], gamma)

    # Wave speeds
    SL = min(qL[1] - cL, qR[1] - cR)
    SR = max(qL[1] + cL, qR[1] + cR)

    return SL, SR

### 3. Lax-Friedrichs:

This solver usis the average of the 2 fluxes, with a diffusion term that uses the fastest speed of the two states:

$$\mathbf{F}^{\rm LF} = \frac{1}{2}\left(\mathbf{F}_L + \mathbf{F}_R\right) - \frac{1}{2}S_{\rm max}\left(\mathbf{U}_R - \mathbf{U}_L\right), \qquad S_{\rm max} = \max\left(\left|u_L\right| + c_L,\ \left|u_R\right| + c_R\right)$$

In [ ]:
def riemann_lf(qL, qR, gamma = 1.4):
    """
    Lax-Friedrichs (Rusanov) flux.
    """

    # Left and Right Conservaed
    UL = conserved(qL, gamma)
    UR = conserved(qR, gamma)

    # Sound speeds
    cL = sound_speed(qL[0], qL[2], gamma)
    cR = sound_speed(qR[0], qR[2], gamma)

    # Max. wave speed
    Smax = max(abs(qL[1]) + cL, abs(qR[1]) + cR)

    # Centred flux minus a diffusion term acting on jump
    rus_lf = 0.5*(flux(qL, gamma) + flux(qR, gamma)) - 0.5*Smax*(UR - UL)

    return rus_lf

### 4. HLL solver:

This solver uses 2 waves and one constant state between them.

The state follows from conservation over the fan, and the flux from the jump condition across either outer wave:

$$\mathbf{F}^{\rm hll} = \begin{cases}\mathbf{F}_L & 0 \leq S_L\\[6pt] \dfrac{S_R\mathbf{F}_L - S_L\mathbf{F}_R + S_LS_R\left(\mathbf{U}_R - \mathbf{U}_L\right)}{S_R - S_L} & S_L \leq 0 \leq S_R\\[10pt] \mathbf{F}_R & 0 \geq S_R\end{cases}$$

In [ ]:
def riemann_hll(qL, qR, gamma = 1.4):
    """
    HLL flux: 2 waves + 1 intermediate state.
    """
    # Signal speeds
    SL, SR = signal_speeds(qL, qR, gamma)

    # L and R fluxes
    FL = flux(qL, gamma)
    FR = flux(qR, gamma)

    # The fan leaves to the right, so the interface keeps the left state
    if SL >= 0.:
        return FL

    # The fan leaves to the left, so the interface keeps the right state
    if SR <= 0.:
        return FR

    # Back to conserved
    UL = conserved(qL, gamma)
    UR = conserved(qR, gamma)

    # See eq. above
    f_hll = (SR*FL - SL*FR + SL*SR*(UR - UL))/(SR - SL)
    
    return f_hll

### 5. HLLC:

In this solver, the contact is added as a third wave, with pressure and velocity continuous across it. 

Its speed is:

$$S_{\star} = \frac{p_R - p_L + \rho_Lu_L\left(S_L - u_L\right) - \rho_Ru_R\left(S_R - u_R\right)}{\rho_L\left(S_L - u_L\right) - \rho_R\left(S_R - u_R\right)}$$

and the state behind the outer wave of side $K$ is:

$$\mathbf{U}_{\star K} = \rho_K\frac{S_K - u_K}{S_K - S_{\star}}\begin{pmatrix}1\\ S_{\star}\\ \frac{E_K}{\rho_K} + \left(S_{\star} - u_K\right)\left[S_{\star} + \frac{p_K}{\rho_K\left(S_K - u_K\right)}\right]\end{pmatrix}, \qquad\qquad \mathbf{F}_{\star K} = \mathbf{F}_K + S_K\left(\mathbf{U}_{\star K} - \mathbf{U}_K\right)$$

In [ ]:
def riemann_hllc(qL, qR, gamma = 1.4):
    """
    HLLC flux: two outer waves and the contact.
    """
    # L and R signal speeds
    SL, SR = signal_speeds(qL, qR, gamma)

    # Fluxes
    FL = flux(qL, gamma)
    FR = flux(qR, gamma)

    # Same two cases as in HLL
    if SL >= 0.:
        return FL

    if SR <= 0.:
        return FR
    
    # Primitive quantities
    rhoL, uL, pL = qL
    rhoR, uR, pR = qR

    # Speed of the contact
    Sstar = (pR - pL + rhoL*uL*(SL - uL) - rhoR*uR*(SR - uR))/(rhoL*(SL - uL) - rhoR*(SR - uR))

    def star_state(q, S, U):
        """
        The state between the outer wave of speed S and the contact.
        """

        # get variables
        rho, u, p = q

        # Density behind the outer wave, from mass conservation across it
        factor = rho*(S - u)/(S - Sstar)

        # (rho*, rho*.S*, E*), energy from the jump condition
        s_state = factor*np.array([1., Sstar, U[2]/rho + (Sstar - u)*(Sstar + p/(rho*(S - u)))])

        return s_state

    # The contact moves to the right, so the interface sees the left star state
    if Sstar >= 0.:
        UL = conserved(qL, gamma)
        return FL + SL*(star_state(qL, SL, UL) - UL)

    # Otherwise it sees the right star state

    # Back to conserved
    UR = conserved(qR, gamma)

    f_hllc = FR + SR*(star_state(qR, SR, UR) - UR)

    return f_hllc

### 6. Roe:

In this solver, the system is linearised about the density-weighted averages:

$$\tilde{u} = \frac{\sqrt{\rho_L}\,u_L + \sqrt{\rho_R}\,u_R}{\sqrt{\rho_L} + \sqrt{\rho_R}}, \qquad \tilde{H} = \frac{\sqrt{\rho_L}\,H_L + \sqrt{\rho_R}\,H_R}{\sqrt{\rho_L} + \sqrt{\rho_R}}, \qquad \tilde{c} = \sqrt{\left(\gamma - 1\right)\left(\tilde{H} - \tfrac{1}{2}\tilde{u}^{2}\right)}$$

with $H = (E + p)/\rho$, and the jump is carried wave by wave:

$$\mathbf{F}^{\rm roe} = \frac{1}{2}\left(\mathbf{F}_L + \mathbf{F}_R\right) - \frac{1}{2}\sum_{\nu}\left|\tilde{\lambda}^{(\nu)}\right|\tilde{\alpha}^{(\nu)}\tilde{\mathbf{r}}^{(\nu)}$$

- Harten's entropy fix is applied to the 2 acoustic waves, where a speed can cross zero inside a rarefaction. 

- The contact is left alone, so that it stays sharp.

In [ ]:
def riemann_roe(qL, qR, gamma = 1.4, eps = 0.1):
    """
    Roe flux, with an entropy fix on the two acoustic waves.
    """
    # L and R primitive vars.
    rhoL, uL, pL = qL
    rhoR, uR, pR = qR

    # Conserved q's
    UL = conserved(qL, gamma)
    UR = conserved(qR, gamma)

    # Specific total enthalpies
    HL = (UL[2] + pL)/rhoL
    HR = (UR[2] + pR)/rhoR

    # Roe averages
    sL = np.sqrt(rhoL)
    sR = np.sqrt(rhoR)

    # Roe-averaged density, velocity, enthalpy and sound speed
    rho = sL*sR
    u = (sL*uL + sR*uR)/(sL + sR)
    H = (sL*HL + sR*HR)/(sL + sR)
    c = np.sqrt((gamma - 1.)*(H - 0.5*u**2))

    # Eigenvalues and wave amplitudes
    lam = np.array([u - c, u, u + c])

    # Jumps in the primitive variables across the interface
    dp = pR - pL
    du = uR - uL
    drho = rhoR - rhoL

    # Amplitudes of the three waves: the jump projected on the eigenvectors
    alpha = np.array([(dp - rho*c*du)/(2.*c**2), drho - dp/c**2, (dp + rho*c*du)/(2.*c**2)])

    # Right eigenvectors -> one per column
    r = np.array([[1., 1., 1.], [u - c, u, u + c], [H - u*c, 0.5*u**2, H + u*c]])

    # Entropy fix on the acoustic waves
    absl = np.abs(lam)
    small = eps*c

    # Only the acoustic waves (nu = 0, nu= 2) are fixed
    for nu in (0, 2):
        if absl[nu] < small:
            absl[nu] = (lam[nu]**2 + small**2)/(2.*small)

    # Centred flux minus the diffusion (applied wave by wave)
    f_roe = 0.5*(flux(qL, gamma) + flux(qR, gamma)) - 0.5*r @ (absl*alpha)
    
    return f_roe

## Part II. The Sod problem:

### 1. The finite-volume solver:

Piecewise constant reconstruction, so that the only difference between the runs is the Riemann solver.

In [ ]:
def solve(nx, riemann, tend, qL, qR, x0 = 0.5, cfl = 0.8, gamma = 1.4):
    """
    Finite-volume solution of a Riemann problem on nx cells.
    """

    # Ghost cells, and the first and last real cell
    ng = 2
    ilo, ihi = ng, ng + nx - 1

    # Cell centres, ghosts included
    dx = 1./nx
    x = (np.arange(nx + 2*ng) - ng + 0.5)*dx

    # Initial data
    q = np.zeros((3, nx + 2*ng))
    q[:, x < x0] = np.array(qL)[:, None]
    q[:, x >= x0] = np.array(qR)[:, None]

    U = conserved(q, gamma)

    t = 0.

    while t < tend:

        # Outflow boundaries
        for k in range(ng):
            U[:, ilo - 1 - k] = U[:, ilo]
            U[:, ihi + 1 + k] = U[:, ihi]

        q = primitive(U, gamma)

        # Timestep from the CFL condition, cut short at the final time
        c = sound_speed(q[0], q[2], gamma)
        dt = min(cfl*dx/np.max(np.abs(q[1]) + c), tend - t)

        # One Riemann problem per interface
        F = np.zeros((3, nx + 2*ng))

        for i in range(ilo, ihi + 2):
            F[:, i] = riemann(q[:, i-1], q[:, i], gamma)

        # Conservative update
        U[:, ilo:ihi+1] += dt/dx*(F[:, ilo:ihi+1] - F[:, ilo+1:ihi+2])

        t += dt

    # Interior cells only
    return x[ilo:ihi+1], primitive(U, gamma)[:, ilo:ihi+1]

### 2. The exact solution:

The solver of nb18, used here only as the reference curve.

In [ ]:
def f_K(p, rhoK, uK, pK, gamma):
    """
    One side of the pressure function, and its derivative.
    """

    cK = sound_speed(rhoK, pK, gamma)

    AK = 2./((gamma + 1.)*rhoK)
    BK = (gamma - 1.)/(gamma + 1.)*pK

    # Shock branch
    if p > pK:
        f = (p - pK)*np.sqrt(AK/(p + BK))
        df = np.sqrt(AK/(BK + p))*(1. - 0.5*(p - pK)/(BK + p))

    # Rarefaction branch
    else:
        f = 2.*cK/(gamma - 1.)*((p/pK)**((gamma - 1.)/(2.*gamma)) - 1.)
        df = 1./(rhoK*cK)*(p/pK)**(-(gamma + 1.)/(2.*gamma))

    return f, df


def star_state(qL, qR, gamma = 1.4, tol = 1.e-12):
    """
    Pressure and velocity in the star region, by the Newton-Raphson method.
    """

    rhoL, uL, pL = qL
    rhoR, uR, pR = qR

    # Initial guess
    p = max(tol, 0.5*(pL + pR))

    # Newton-Raphson on f(p) = fL + fR + (uR - uL)
    for it in range(50):

        fL, dfL = f_K(p, rhoL, uL, pL, gamma)
        fR, dfR = f_K(p, rhoR, uR, pR, gamma)

        p_new = max(tol, p - (fL + fR + uR - uL)/(dfL + dfR))

        if abs(p_new - p) < tol*p_new:
            p = p_new
            break

        p = p_new

    fL = f_K(p, rhoL, uL, pL, gamma)[0]
    fR = f_K(p, rhoR, uR, pR, gamma)[0]

    return p, 0.5*(uL + uR) + 0.5*(fR - fL)


def sample(xi, qL, qR, pstar, ustar, gamma = 1.4):
    """
    Exact state at xi = (x - x0)/t
    """

    rhoL, uL, pL = qL
    rhoR, uR, pR = qR

    cL = sound_speed(rhoL, pL, gamma)
    cR = sound_speed(rhoR, pR, gamma)

    # Left of the contact
    if xi <= ustar:

        # Left wave is a shock
        if pstar > pL:

            S = uL - cL*np.sqrt((gamma + 1.)/(2.*gamma)*pstar/pL + (gamma - 1.)/(2.*gamma))

            if xi < S:
                return rhoL, uL, pL

            ratio = pstar/pL
            rho = rhoL*(ratio + (gamma - 1.)/(gamma + 1.))/((gamma - 1.)/(gamma + 1.)*ratio + 1.)
            return rho, ustar, pstar

        # Left wave is a rarefaction
        c_star = cL*(pstar/pL)**((gamma - 1.)/(2.*gamma))

        if xi < uL - cL:
            return rhoL, uL, pL

        if xi > ustar - c_star:
            return rhoL*(pstar/pL)**(1./gamma), ustar, pstar

        u = 2./(gamma + 1.)*(cL + (gamma - 1.)/2.*uL + xi)
        c = u - xi
        return rhoL*(c/cL)**(2./(gamma - 1.)), u, pL*(c/cL)**(2.*gamma/(gamma - 1.))

    # Right of the contact
    else:

        # Right wave is a shock
        if pstar > pR:

            S = uR + cR*np.sqrt((gamma + 1.)/(2.*gamma)*pstar/pR + (gamma - 1.)/(2.*gamma))

            if xi > S:
                return rhoR, uR, pR

            ratio = pstar/pR
            rho = rhoR*(ratio + (gamma - 1.)/(gamma + 1.))/((gamma - 1.)/(gamma + 1.)*ratio + 1.)
            return rho, ustar, pstar

        # Right wave is a rarefaction
        c_star = cR*(pstar/pR)**((gamma - 1.)/(2.*gamma))

        if xi > uR + cR:
            return rhoR, uR, pR

        if xi < ustar + c_star:
            return rhoR*(pstar/pR)**(1./gamma), ustar, pstar

        u = 2./(gamma + 1.)*(-cR + (gamma - 1.)/2.*uR + xi)
        c = xi - u
        return rhoR*(c/cR)**(2./(gamma - 1.)), u, pR*(c/cR)**(2.*gamma/(gamma - 1.))


def exact(x, t, qL, qR, x0 = 0.5, gamma = 1.4):
    """
    The exact solution on a grid of x at time t.
    """

    ps, us = star_state(qL, qR, gamma)

    return np.array([sample((xk - x0)/t, qL, qR, ps, us, gamma) for xk in x]).T

### 3. Comparison of solvers on the shock tube:

In [ ]:
# Name, function and colour of each solver
SOLVERS = [("Lax-Friedrichs", riemann_lf, "grey"),("HLL", riemann_hll, "tab:blue"),\
           ("HLLC", riemann_hllc, "tab:green"), ("Roe", riemann_roe, "tab:red")]

# Sod's initial states, q = (rho, u, p)
qL_sod = (1.0, 0.0, 1.0)
qR_sod = (0.125, 0.0, 0.1)

# Resolution and final time
nx = 128
tend = 0.25

# One run per solver, everything else identical
runs_sod = {}

for name, riemann, colour in SOLVERS:
    runs_sod[name] = solve(nx, riemann, tend, qL_sod, qR_sod)

In [ ]:
def plot_runs(runs, qL, qR, t, gamma = 1.4, xlim = (0., 1.)):
    """
    Density, velocity, pressure and internal energy of several runs.
    """

    # Exact solution on a fine grid, plus its internal energy
    x_fine = np.linspace(0., 1., 1000)
    qe = exact(x_fine, t, qL, qR, gamma = gamma)
    ee = qe[2]/((gamma - 1.)*qe[0])

    fig, ax = plt.subplots(4, 1, figsize = (7,12), sharex = True)

    # One panel per variable, with the exact solution as a black line
    for axis, var, lab in zip(ax, [qe[0], qe[1], qe[2], ee], [r"$\rho$", r"$u$", r"$p$", r"$e$"]):
        axis.plot(x_fine, var, color = "black", lw = 1.5, label = "exact")
        axis.set_ylabel(lab)
        axis.grid(alpha = 0.3)

    # The runs on top, one colour per solver
    for name, riemann, colour in SOLVERS:

        x, q = runs[name]
        e = q[2]/((gamma - 1.)*q[0])

        for axis, var in zip(ax, [q[0], q[1], q[2], e]):
            axis.plot(x, var, "o", color = colour, ms = 3., label = name)

    ax[0].legend(loc = "upper right")
    ax[3].set_xlabel(r"$x$")
    plt.xlim(*xlim)
    plt.tight_layout()
    plt.show()

### Observations:

- All solvers place the rarefaction, the contact, and the shock at the same positions. 

- The differences are in the width of the contac and in the overshoot of the internal energy there.

### 4. Errors and convergence:

The $L_1$ error in density against the exact solution, at four resolutions.

In [ ]:
def l1_error(x, q, qL, qR, t, gamma = 1.4):
    """
    L1 error in density against the exact solution, on the grid of the run.
    """

    qe = exact(x, t, qL, qR, gamma = gamma)

    return np.mean(np.abs(q[0] - qe[0]))


# Four resolutions, same data and same final time
RESOLUTIONS = [64, 128, 256, 512]

# One run per solver and per resolution
errors = {}

for name, riemann, colour in SOLVERS:

    errors[name] = []

    for n in RESOLUTIONS:
        x, q = solve(n, riemann, tend, qL_sod, qR_sod)
        errors[name].append(l1_error(x, q, qL_sod, qR_sod, tend))

In [ ]:
plot_runs(runs_sod, qL_sod, qR_sod, tend)

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4.6))

# The error of each solver against resolution
for name, riemann, colour in SOLVERS:
    ax.loglog(RESOLUTIONS, errors[name], "o-", color = colour, label = name)

# First-order reference slope
n = np.array(RESOLUTIONS, dtype = float)
ax.loglog(n, errors["HLLC"][0]*(n/n[0])**-1., "k--", lw = 1., label = r"$n_x^{-1}$")

ax.set_xscale("log", base = 2)
ax.set_xticks(RESOLUTIONS)
ax.set_xticklabels([str(v) for v in RESOLUTIONS])
ax.minorticks_off()
ax.set_xlabel(r"$n_x$")
ax.set_ylabel(r"$E_1$")
ax.legend(frameon = False, fontsize = 9)
ax.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

## Which solver should I use?

- The 4 solvers are **consistent**: they put the rarefaction, the contact, and the shock in the same places. They differ only in how much they smear the contact.

-  They also converge at the same rate. A better solver lowers the error, but does not raise the order.

- **TVD Lax-Friedrichs**: one diffusion coefficient for every field, so the contact is as damped as the shock. Cheapest and the most robust. Use it as a fallback, for very strong shocks and near vacuum.

- **HLL**: 2 waves and 1 state between them, no contact. Positive and cheap. Use it when the flow is dominated by shocks and robustness matters more than the contact.

- **HLLC**: the contact restored for one division more than HLL. Complete, accurate and still positive. **Default choice for the Euler equations.**

- **Roe**: sharpest because the diffusion is applied wave by wave. It needs an entropy fix, and it can return a negative pressure in a strong expansion. Use it for smooth or weakly non-linear flows, and where contacts and shear layers must stay sharp.

- Rule of thumb: **run HLLC everywhere, and fall back to HLL or Lax-Friedrichs only at the interfaces where it fails.**
